# Step 1: FP8 量化（W8A8，llm-compressor，H200 首选）

**目标**：用 `llmcompressor` 的 `QuantizationModifier` 把 Qwen2.5-7B-Instruct 量化成 **FP8（E4M3）** 格式——权重 per-channel 静态、激活 dynamic per-token，**无需校准数据**，产物为 compressed-tensors 格式，可被 vLLM 直接加载。

**对应 OUTLINE 课时**：2.3 FP8 量化全流程（~45 分钟）。

> FP8 是 H200（sm90）上的首选：Hopper 原生 FP8 张量核、无需校准、精度损失最小。本步是整条流水线里最简单的一步——掌握 `QuantizationModifier` 的 `targets` / `scheme` / `ignore` 三参数即可。

In [ ]:
%%capture
import subprocess, pathlib, json, shutil
import torch
import ipytest
ipytest.autoconfig()
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

In [ ]:
# 解析仓库根目录（cwd 无关）：notebooks 通过 `uv run --directory envs/quant jupyter lab`
# 启动，但 jupyter 的 cwd 是所在 shell 的 cwd（不是 --directory 目标），所以
# 所有路径都从 git 仓库根派生，绝不依赖裸相对路径。
import subprocess, pathlib

REPO_ROOT = pathlib.Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)
MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 download_model.sh 默认一致
TINY_MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-0.5B-Instruct"  # L3 先在 0.5B 上验，再上 7B
OUT_ROOT = REPO_ROOT / "out"                                  # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("GPU OK" if __import__("torch").cuda.is_available() else "无 GPU（仅 L1/L2 可跑）")

## 原理：FP8_DYNAMIC 是什么

`scheme="FP8_DYNAMIC"`（llmcompressor 当前版本确切 API，**不是**已废弃的 `Float8Modifier`）的含义：

| 维度 | 量化 | 说明 |
|------|------|------|
| 权重 (W) | E4M3，**per-channel**，**静态** | 每个 weight channel 一个 scale，离线算好烤进权重 |
| 激活 (A) | E4M3，**per-token**，**dynamic** | 每次推理按当前 token 的激活范围动态算 scale——所以**无需校准数据** |

`targets="Linear"`：只量化所有 `nn.Linear` 子模块；`ignore=["lm_head"]`：跳过输出头（lm_head 对精度敏感且常与 embedding 共享权重，量化会显著掉点）。

**易错点（OUTLINE 标注）**：旧博客写 `from llmcompressor.modifiers.quantization import Float8Modifier`——新版会 `ImportError`，必须用 `QuantizationModifier(scheme="FP8_DYNAMIC")`。

## 本步填空

你要实现两个 `logic` 函数，逐步搭出 FP8 量化能力：

1. **`build_fp8_recipe(ignore)`** —— 构造 `QuantizationModifier`（核心：scheme/ targets/ ignore 三参数）。
2. **`extract_quant_summary(qc)`** —— 从 `config.json` 的 `quantization_config` 里抽出可断言的关键字段（教学：学会读 compressed-tensors 结构，为产物检查打基础）。

填完每个函数后跑紧随其后的 `%%ipytest` cell 验证（L1）。

In [ ]:
def build_fp8_recipe(ignore=("lm_head",)):
    """返回一个 FP8 动态量化的 QuantizationModifier。

    要求：
      - targets="Linear"            （只量化 Linear 层）
      - scheme="FP8_DYNAMIC"        （权重 per-channel 静态 + 激活 per-token 动态 E4M3）
      - ignore=list(ignore)         （跳过的模块名，默认 lm_head；必须是 list）
    """
    # TODO: 用 llmcompressor.modifiers.quantization.QuantizationModifier 构造并返回
    #       提示：from llmcompressor.modifiers.quantization import QuantizationModifier
    #             QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC", ignore=list(ignore))
    raise NotImplementedError


# 脚手架（提供）：真正把模型跑成 FP8 的 execution 函数，调用你填好的 recipe。
def run_fp8_quantize(model, save_dir):
    recipe = build_fp8_recipe()
    # 注意：FP8 的 oneshot 不传 dataset（动态激活量化，无需校准）
    oneshot(model=model, recipe=recipe)
    save_dir = pathlib.Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    return save_dir

In [ ]:
def extract_quant_summary(quantization_config):
    """从 config.json 的 quantization_config 字典抽出关键断言字段。

    返回一个 dict，至少含：
      - "quant_method"        : str   （应为 "compressed-tensors"）
      - "weights_num_bits"    : int   （权重位宽）
      - "weights_type"        : str   （权重类型，如 "float"）
      - "input_dynamic"       : bool  （激活是否动态量化）
      - "targets"             : list  （被量化的模块，如 ["Linear"]）
      - "ignore"              : list  （被忽略的模块名）
    compressed-tensors 结构提示：
      quantization_config["config_groups"]["group_0"]["weights"]["num_bits"] 等
      quantization_config["config_groups"]["group_0"]["input_activations"]["dynamic"]
      quantization_config["config_groups"]["group_0"]["targets"]
      quantization_config["ignore"]
    """
    # TODO: 解析 quantization_config，返回上述 dict
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_build_fp8_recipe_scheme_and_targets():
    r = build_fp8_recipe()
    assert r.scheme == "FP8_DYNAMIC"
    # llmcompressor 把 targets 标准化成 list（字符串 "Linear" -> ["Linear"]）
    assert list(r.targets) == ["Linear"]

def test_build_fp8_recipe_ignores_lm_head_default():
    r = build_fp8_recipe()
    assert "lm_head" in r.ignore
    assert isinstance(r.ignore, list)   # OUTLINE 提醒：ignore 必须是 list，写字符串会报错

def test_build_fp8_recipe_custom_ignore():
    r = build_fp8_recipe(ignore=("lm_head", "re:visual"))
    assert r.ignore == ["lm_head", "re:visual"]

def test_extract_quant_summary_on_fake_fp8():
    fake = {
        "quant_method": "compressed-tensors",
        "ignore": ["lm_head"],
        "config_groups": {"group_0": {
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "float"},
            "input_activations": {"dynamic": True, "num_bits": 8},
        }},
    }
    s = extract_quant_summary(fake)
    assert s["quant_method"] == "compressed-tensors"
    assert s["weights_num_bits"] == 8
    assert s["weights_type"] == "float"
    assert s["input_dynamic"] is True
    assert s["targets"] == ["Linear"]
    assert s["ignore"] == ["lm_head"]

## L2：tiny 模型验证（CPU/GPU 秒级）

用内存里随机初始化的 **tiny Qwen2**（2 层、hidden 128）真跑 llmcompressor 的 FP8 流水线——这是 C2（早发现框架问题）的关键：API 误用 / recipe 结构错在 tiny 规模就暴露，不用等上 7B。

In [ ]:
# tiny 模型 + 配套 tokenizer（词表对齐，避免越界）
from transformers import Qwen2Config, Qwen2ForCausalLM

def make_tiny_model(vocab_size=512, hidden_size=128):
    cfg = Qwen2Config(
        num_hidden_layers=2, hidden_size=hidden_size, intermediate_size=hidden_size * 2,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=vocab_size,
        tie_word_embeddings=True,
    )
    m = Qwen2ForCausalLM(cfg).eval()
    return m

tiny = make_tiny_model()
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny.to(device)
print("tiny 模型就绪:", sum(p.numel() for p in tiny.parameters()), "params, device:", device)

# 用你填好的函数真跑 FP8（tiny 规模）
tiny_out = OUT_ROOT / "tiny-fp8"
run_fp8_quantize(tiny, tiny_out)

qc = json.loads((tiny_out / "config.json").read_text())["quantization_config"]
summary = extract_quant_summary(qc)
print("FP8 产物摘要:", summary)
assert summary["quant_method"] == "compressed-tensors"
assert summary["weights_num_bits"] == 8
assert summary["input_dynamic"] is True
print("L2 PASS：tiny FP8 流水线跑通，产物结构正确")

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B）

GPU 守卫：无 GPU（CPU 环境）自动跳过，只跑 L1/L2。本机若是 H200/L20X（sm90）则真跑——先 0.5B 快验，再 7B 出可部署产物。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM

    # 先 0.5B 快验（秒级，确认真模型也能跑通）
    model_05b = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_05b = OUT_ROOT / "qwen05b-fp8"
    run_fp8_quantize(model_05b, out_05b)
    print("0.5B FP8 done ->", out_05b)
    del model_05b; torch.cuda.empty_cache()

    # 再 7B（出可部署产物）
    model_7b = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_7b = OUT_ROOT / "qwen7b-fp8"
    run_fp8_quantize(model_7b, out_7b)
    print("7B FP8 done ->", out_7b)
    del model_7b; torch.cuda.empty_cache()
else:
    print("跳过：无 GPU（CPU 环境只跑 L1/L2）。L2 已证明逻辑 + 库集成 OK。")

## 产物检查

打印 7B FP8 产物的 `quantization_config` 与显存占用对比（FP16 vs FP8）。

In [ ]:
def report_artifact(out_dir):
    out_dir = pathlib.Path(out_dir)
    if not out_dir.exists():
        print(f"(跳过：{out_dir} 不存在，可能 L3 未跑)")
        return None
    cfg = json.loads((out_dir / "config.json").read_text())
    qc = cfg.get("quantization_config", {})
    total_bytes = sum(f.stat().st_size for f in out_dir.glob("*.safetensors"))
    print(f"== {out_dir.name} ==")
    print("  quant_method      :", qc.get("quant_method"))
    print("  ignore            :", qc.get("ignore"))
    g0 = list(qc.get("config_groups", {}).values())
    if g0:
        w, a = g0[0]["weights"], g0[0].get("input_activations")
        print(f"  weights           : {w['num_bits']}-bit {w['type']} strategy={w['strategy']}")
        print(f"  input_activations : {a['num_bits']}-bit dynamic={a['dynamic']} strategy={a['strategy']}" if a else "  input_activations : None")
    print(f"  safetensors 总大小: {total_bytes/1e9:.2f} GB")
    return total_bytes

sizes = {}
sizes["FP8 7B"] = report_artifact(OUT_ROOT / "qwen7b-fp8")

# 显存/磁盘对比：FP16 原模型
fp16_bytes = sum(f.stat().st_size for f in MODEL_DIR.glob("*.safetensors"))
sizes["FP16 7B (原始)"] = fp16_bytes
print(f"\n== 原始 FP16 7B safetensors: {fp16_bytes/1e9:.2f} GB ==")
if sizes.get("FP8 7B"):
    ratio = sizes["FP8 7B"] / fp16_bytes
    print(f"== FP8/FP16 磁盘比: {ratio:.2%}（FP8 权重理论约 50%，略高于此因含 bf16 lm_head/非量化部分）==")